In [1]:
import json
import os

import pandas as pd
import psycopg
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

conn = psycopg.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=os.getenv("POSTGRES_PORT", "5436"),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

with conn.cursor() as cur:
    cur.execute("""
        SELECT id, start_date, end_date, http_status, payload, ingested_at
        FROM bronze.neows_feed_raw
        ORDER BY id
    """)
    cols = [c.name for c in cur.description]
    df = pd.DataFrame(cur.fetchall(), columns=cols)

df.shape

(4, 6)

In [2]:
df["element_count"] = df["payload"].apply(lambda p: p["element_count"])
df[["id", "start_date", "end_date", "http_status", "element_count", "ingested_at"]]

,id,start_date,end_date,http_status,element_count,ingested_at
0,1,2026-11-15,2026-11-21,200,32,2026-11-15 23:59:11.608481+00:00
1,3,2026-11-16,2026-11-22,200,33,2026-11-16 22:43:35.854302+00:00
2,4,2026-11-16,2026-11-22,200,33,2026-11-16 22:58:39.240941+00:00
3,5,2026-11-17,2026-11-23,200,32,2026-11-17 03:34:04.836753+00:00


In [3]:
payload = df.loc[df["id"].idxmax(), "payload"]

print("Chaves do topo:", list(payload.keys()))
print("Datas no feed:", list(payload["near_earth_objects"].keys()))

primeira_data = next(iter(payload["near_earth_objects"]))
asteroide = payload["near_earth_objects"][primeira_data][0]
print(json.dumps(asteroide, indent=2)[:3000])

Chaves do topo: ['links', 'element_count', 'near_earth_objects']
Datas no feed: ['2026-11-17', '2026-11-18', '2026-11-19', '2026-11-20', '2026-11-21', '2026-11-22', '2026-11-23']
{
  "id": "2322756",
  "name": "322756 (2001 CK32)",
  "links": {
    "self": "http://api.nasa.gov/neo/rest/v1/neo/2322756?api_key=REDACTED"
  },
  "nasa_jpl_url": "https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2322756",
  "is_sentry_object": false,
  "neo_reference_id": "2322756",
  "estimated_diameter": {
    "feet": {
      "estimated_diameter_max": 2779.8734775794,
      "estimated_diameter_min": 1243.1972129433
    },
    "miles": {
      "estimated_diameter_max": 0.5264910092,
      "estimated_diameter_min": 0.2354539372
    },
    "meters": {
      "estimated_diameter_max": 847.3054088524,
      "estimated_diameter_min": 378.9264983795
    },
    "kilometers": {
      "estimated_diameter_max": 0.8473054089,
      "estimated_diameter_min": 0.3789264984
    }
  },
  "close_approach_data": [
    {

In [4]:
linhas = []
for _, r in df.iterrows():
    for data_feed, asteroides in r["payload"]["near_earth_objects"].items():
        for a in asteroides:
            linhas.append({**a, "_feed_date": data_feed, "_bronze_id": r["id"]})

asteroides_df = pd.json_normalize(linhas, sep="_").drop(columns=["close_approach_data"])

aproximacoes_df = pd.json_normalize(
    linhas,
    record_path="close_approach_data",
    meta=["id", "_bronze_id"],
    sep="_",
)

print(asteroides_df.shape, aproximacoes_df.shape)
asteroides_df.columns.tolist()

(130, 19) (130, 13)


['id',
 'name',
 'nasa_jpl_url',
 'is_sentry_object',
 'neo_reference_id',
 'absolute_magnitude_h',
 'is_potentially_hazardous_asteroid',
 '_feed_date',
 '_bronze_id',
 'links_self',
 'estimated_diameter_feet_estimated_diameter_max',
 'estimated_diameter_feet_estimated_diameter_min',
 'estimated_diameter_miles_estimated_diameter_max',
 'estimated_diameter_miles_estimated_diameter_min',
 'estimated_diameter_meters_estimated_diameter_max',
 'estimated_diameter_meters_estimated_diameter_min',
 'estimated_diameter_kilometers_estimated_diameter_max',
 'estimated_diameter_kilometers_estimated_diameter_min',
 'sentry_data']

In [5]:
numericas = [
    "relative_velocity_kilometers_per_second",
    "miss_distance_astronomical",
    "miss_distance_lunar",
    "miss_distance_kilometers",
]

convertidas = aproximacoes_df[numericas].apply(pd.to_numeric, errors="coerce")

# se alguma conversão virou nulo, o dado original tinha algo inesperado
print(convertidas.isna().sum())
print(aproximacoes_df[numericas].isna().sum())  # nulos que já existiam

datas = pd.to_datetime(
    aproximacoes_df["close_approach_date_full"], format="%Y-%b-%d %H:%M", errors="coerce"
)
print(datas.isna().sum(), "datas não convertidas")

relative_velocity_kilometers_per_second    0
miss_distance_astronomical                 0
miss_distance_lunar                        0
miss_distance_kilometers                   0
dtype: int64
relative_velocity_kilometers_per_second    0
miss_distance_astronomical                 0
miss_distance_lunar                        0
miss_distance_kilometers                   0
dtype: int64
0 datas não convertidas


In [6]:
print("Asteroides repetidos (por id):", asteroides_df["id"].duplicated().sum())
print(
    "Aproximações repetidas:",
    aproximacoes_df.duplicated(subset=["id", "epoch_date_close_approach", "orbiting_body"]).sum(),
)

asteroides_df.isna().sum()[lambda s: s > 0]

diam = [c for c in asteroides_df.columns if c.startswith("estimated_diameter_meters")]
asteroides_df[diam + ["absolute_magnitude_h"]].describe()

Asteroides repetidos (por id): 95
Aproximações repetidas: 95


,estimated_diameter_meters_estimated_diameter_max,estimated_diameter_meters_estimated_diameter_min,absolute_magnitude_h
count,130.000000,130.000000,130.000000
mean,218.351881,97.649930,23.549231
std,237.976070,106.426134,2.735743
min,8.995804,4.023046,18.800000
25%,51.935386,23.226211,21.400000
50%,143.893103,64.350952,23.080000
75%,311.917671,139.493823,25.300000
max,1032.856481,461.907460,29.100000


In [7]:
# shape = (número de linhas, número de colunas)
print("asteroides:", asteroides_df.shape)
print("aproximações:", aproximacoes_df.shape)

# colunas do dataframe de aproximações
print(list(aproximacoes_df.columns))

# quantos ids de asteroide se repetem (duplicatas)
print("ids repetidos:", asteroides_df["id"].duplicated().sum())

asteroides: (130, 19)
aproximações: (130, 13)
['orbiting_body', 'close_approach_date', 'close_approach_date_full', 'epoch_date_close_approach', 'miss_distance_lunar', 'miss_distance_miles', 'miss_distance_kilometers', 'miss_distance_astronomical', 'relative_velocity_miles_per_hour', 'relative_velocity_kilometers_per_hour', 'relative_velocity_kilometers_per_second', 'id', '_bronze_id']
ids repetidos: 95


In [8]:
# --- APROXIMAÇÕES ---

# colunas que vieram como texto mas são números
colunas_numericas = [
    "miss_distance_lunar",
    "miss_distance_miles",
    "miss_distance_kilometers",
    "miss_distance_astronomical",
    "relative_velocity_miles_per_hour",
    "relative_velocity_kilometers_per_hour",
    "relative_velocity_kilometers_per_second",
]

# pd.to_numeric converte texto em número.
# errors="coerce": se algum valor não for conversível, vira NaN (vazio)
# em vez de dar erro. Assim a checagem de qualidade depois consegue achar o problema.
for coluna in colunas_numericas:
    aproximacoes_df[coluna] = pd.to_numeric(aproximacoes_df[coluna], errors="coerce")

# data simples: texto "2026-11-16" vira data de verdade
aproximacoes_df["close_approach_date"] = pd.to_datetime(aproximacoes_df["close_approach_date"])

# data com hora: epoch em milissegundos -> timestamp em UTC
aproximacoes_df["close_approach_ts"] = pd.to_datetime(
    aproximacoes_df["epoch_date_close_approach"], unit="ms", utc=True
)

# --- ASTEROIDES ---

# a data do feed também vira data de verdade
asteroides_df["_feed_date"] = pd.to_datetime(asteroides_df["_feed_date"])

# --- CONFERÊNCIA ---
# dtypes = o tipo de cada coluna (float64 = decimal, object = texto, etc.)
print(aproximacoes_df.dtypes)
print()
print(asteroides_df.dtypes)

orbiting_body                                              str
close_approach_date                             datetime64[us]
close_approach_date_full                                   str
epoch_date_close_approach                                int64
miss_distance_lunar                                    float64
miss_distance_miles                                    float64
miss_distance_kilometers                               float64
miss_distance_astronomical                             float64
relative_velocity_miles_per_hour                       float64
relative_velocity_kilometers_per_hour                  float64
relative_velocity_kilometers_per_second                float64
id                                                         str
_bronze_id                                              object
close_approach_ts                          datetime64[us, UTC]
dtype: object

id                                                                 str
name                            

In [9]:
# converte _bronze_id para inteiro no dataframe de aproximações
aproximacoes_df["_bronze_id"] = aproximacoes_df["_bronze_id"].astype("int64")

# confere os dois lado a lado
print(aproximacoes_df["_bronze_id"].dtype)
print(asteroides_df["_bronze_id"].dtype)

# olha os valores de sentry_data (quais textos existem e quantas vezes)
print(asteroides_df["sentry_data"].value_counts(dropna=False))

int64
int64
sentry_data
NaN                                                                    122
http://api.nasa.gov/neo/rest/v1/neo/sentry/3836296?api_key=REDACTED      4
http://api.nasa.gov/neo/rest/v1/neo/sentry/3843278?api_key=REDACTED      4
Name: count, dtype: int64


In [10]:
# 1) NULOS: quantos valores vazios existem em cada coluna?
#    isna() marca True onde está vazio; sum() conta os True por coluna.
print("Nulos em asteroides (só colunas com algum nulo):")
nulos_ast = asteroides_df.isna().sum()
print(nulos_ast[nulos_ast > 0])

print("\nNulos em aproximações (só colunas com algum nulo):")
nulos_apr = aproximacoes_df.isna().sum()
print(nulos_apr[nulos_apr > 0])

# 2) CONSISTÊNCIA DO SENTRY: is_sentry_object deve ser True
#    exatamente nos asteroides que têm link em sentry_data
print("\nis_sentry_object = True:", asteroides_df["is_sentry_object"].sum())
print("sentry_data preenchido: ", asteroides_df["sentry_data"].notna().sum())

# 3) INTEGRIDADE DA FK: todo id de aproximação existe em asteroides?
#    isin() verifica se cada valor está dentro de outra coluna.
ids_orfaos = ~aproximacoes_df["id"].isin(asteroides_df["id"])
print("\nAproximações sem asteroide correspondente:", ids_orfaos.sum())

# 4) VALORES ESTRANHOS: describe() mostra mínimo, máximo e média
#    das colunas numéricas, para a gente procurar números impossíveis.
colunas_check = [
    "miss_distance_kilometers",
    "relative_velocity_kilometers_per_second",
]
print("\nAproximações:")
print(aproximacoes_df[colunas_check].describe())

print("\nAsteroides:")
print(
    asteroides_df[
        [
            "absolute_magnitude_h",
            "estimated_diameter_meters_estimated_diameter_min",
            "estimated_diameter_meters_estimated_diameter_max",
        ]
    ].describe()
)

# 5) LÓGICA: o diâmetro mínimo nunca pode ser maior que o máximo
min_maior = (
    asteroides_df["estimated_diameter_meters_estimated_diameter_min"]
    > asteroides_df["estimated_diameter_meters_estimated_diameter_max"]
).sum()
print("\nAsteroides com diâmetro min > max:", min_maior)

# 6) CORPO ORBITADO: a NeoWs traz aproximações de vários corpos (Terra, Marte...)?
print("\nCorpos orbitados:")
print(aproximacoes_df["orbiting_body"].value_counts())

Nulos em asteroides (só colunas com algum nulo):
sentry_data    122
dtype: int64

Nulos em aproximações (só colunas com algum nulo):
Series([], dtype: int64)

is_sentry_object = True: 8
sentry_data preenchido:  8

Aproximações sem asteroide correspondente: 0

Aproximações:
       miss_distance_kilometers  relative_velocity_kilometers_per_second
count              1.300000e+02                               130.000000
mean               3.992250e+07                                14.737453
std                2.286287e+07                                 5.050437
min                6.385660e+06                                 3.535095
25%                1.971635e+07                                10.379635
50%                3.844308e+07                                15.029022
75%                6.272024e+07                                18.128678
max                7.397681e+07                                23.166451

Asteroides:
       absolute_magnitude_h  estimated_diameter_meters_e

In [11]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

# --- parte A: conferência que ficou cortada na validação ---
col_min = "estimated_diameter_meters_estimated_diameter_min"
col_max = "estimated_diameter_meters_estimated_diameter_max"
print("menor diâmetro mín (m):", asteroides_df[col_min].min())
print("diâmetro min > max:", (asteroides_df[col_min] > asteroides_df[col_max]).sum())

# --- parte B: chamar o lookup de UM asteroide ---
asteroide_id = asteroides_df["id"].iloc[0]
url = f"https://api.nasa.gov/neo/rest/v1/neo/{asteroide_id}"
resp = requests.get(url, params={"api_key": API_KEY}, timeout=30)
print("\nstatus:", resp.status_code)

# mesma limpeza do extrator: troca a chave no TEXTO antes de virar dicionário
dados = json.loads(resp.text.replace(API_KEY, "REDACTED"))

# chaves do primeiro nível
print("\nchaves principais:", list(dados.keys()))

# o que tem dentro de orbital_data
print("\norbital_data:", list(dados["orbital_data"].keys()))

# quantas aproximações no histórico e de que período
aprox = dados["close_approach_data"]
print("\naproximações:", len(aprox))
print("primeira:", aprox[0]["close_approach_date"])
print("última:  ", aprox[-1]["close_approach_date"])

menor diâmetro mín (m): 4.0230457983
diâmetro min > max: 0

status: 200

chaves principais: ['absolute_magnitude_h', 'close_approach_data', 'designation', 'estimated_diameter', 'id', 'is_potentially_hazardous_asteroid', 'is_sentry_object', 'links', 'name', 'nasa_jpl_url', 'neo_reference_id', 'orbital_data']

orbital_data: ['aphelion_distance', 'ascending_node_longitude', 'data_arc_in_days', 'eccentricity', 'epoch_osculation', 'equinox', 'first_observation_date', 'inclination', 'jupiter_tisserand_invariant', 'last_observation_date', 'mean_anomaly', 'mean_motion', 'minimum_orbit_intersection', 'observations_used', 'orbit_class', 'orbit_determination_date', 'orbit_id', 'orbit_uncertainty', 'orbital_period', 'perihelion_argument', 'perihelion_distance', 'perihelion_time', 'semi_major_axis']

aproximações: 27
primeira: 1930-05-01
última:   2197-05-02


In [12]:
def estrutura(d, prefixo="", nivel=0):
    """Imprime as chaves e os tipos; entra em dicionários e na 1ª posição das listas."""
    for k, v in d.items():
        print(f"{prefixo}{k}: {type(v).__name__}")
        if nivel < 2:
            if isinstance(v, dict):
                estrutura(v, prefixo + "  ", nivel + 1)
            elif isinstance(v, list) and v and isinstance(v[0], dict):
                print(f"{prefixo}  (lista com {len(v)} itens; 1º item:)")
                estrutura(v[0], prefixo + "    ", nivel + 1)


with conn.cursor() as cur:
    cur.execute("SELECT payload FROM bronze.neows_feed_raw ORDER BY id DESC LIMIT 1")
    feed = cur.fetchone()[0]
    cur.execute("SELECT payload FROM bronze.neows_lookup_raw ORDER BY id DESC LIMIT 1")
    lookup = cur.fetchone()[0]

print("=== LOOKUP ===")
estrutura(lookup)

print("\n=== FEED (1º asteroide do 1º dia) ===")
primeiro_dia = next(iter(feed["near_earth_objects"].values()))
estrutura(primeiro_dia[0])

=== LOOKUP ===
id: str
name: str
links: dict
  self: str
designation: str
nasa_jpl_url: str
orbital_data: dict
  equinox: str
  orbit_id: str
  inclination: str
  mean_motion: str
  orbit_class: dict
    orbit_class_type: str
    orbit_class_range: str
    orbit_class_description: str
  eccentricity: str
  mean_anomaly: str
  orbital_period: str
  perihelion_time: str
  semi_major_axis: str
  data_arc_in_days: int
  epoch_osculation: str
  aphelion_distance: str
  observations_used: int
  orbit_uncertainty: str
  perihelion_argument: str
  perihelion_distance: str
  last_observation_date: str
  first_observation_date: str
  ascending_node_longitude: str
  orbit_determination_date: str
  minimum_orbit_intersection: str
  jupiter_tisserand_invariant: str
is_sentry_object: bool
neo_reference_id: str
estimated_diameter: dict
  feet: dict
    estimated_diameter_max: float
    estimated_diameter_min: float
  miles: dict
    estimated_diameter_max: float
    estimated_diameter_min: float
  me

In [13]:
print("LOOKUP:", {k: type(v).__name__ for k, v in lookup.items()})
print("close_approach_data (lookup):", len(lookup["close_approach_data"]), "itens")
print(lookup["close_approach_data"][0], "\n")

ast = primeiro_dia[0]
print("FEED (1 asteroide):", {k: type(v).__name__ for k, v in ast.items()})
print(ast["close_approach_data"][0])

LOOKUP: {'id': 'str', 'name': 'str', 'links': 'dict', 'designation': 'str', 'nasa_jpl_url': 'str', 'orbital_data': 'dict', 'is_sentry_object': 'bool', 'neo_reference_id': 'str', 'estimated_diameter': 'dict', 'close_approach_data': 'list', 'absolute_magnitude_h': 'float', 'is_potentially_hazardous_asteroid': 'bool'}
close_approach_data (lookup): 154 itens
{'miss_distance': {'lunar': '41.7728378597', 'miles': '9982076.6379324638', 'kilometers': '16064595.289631051', 'astronomical': '0.1073851873'}, 'orbiting_body': 'Venus', 'relative_velocity': {'miles_per_hour': '13719.8669627946', 'kilometers_per_hour': '22080.3438088697', 'kilometers_per_second': '6.1334288358'}, 'close_approach_date': '1900-10-09', 'close_approach_date_full': '1900-Oct-09 13:37', 'epoch_date_close_approach': -2184661380000} 

FEED (1 asteroide): {'id': 'str', 'name': 'str', 'links': 'dict', 'nasa_jpl_url': 'str', 'is_sentry_object': 'bool', 'neo_reference_id': 'str', 'estimated_diameter': 'dict', 'close_approach_data

In [ ]:
with conn.cursor() as cur:
    # designações da NeoWs (lookup)
    cur.execute("SELECT payload->>'designation' FROM bronze.neows_lookup_raw")
    des_neows = {linha[0] for linha in cur.fetchall()}

    # designações do CAD (snapshot mais recente da consulta cad_terra)
    cur.execute(
        "SELECT payload FROM bronze.cad_raw WHERE source = 'cad_terra' ORDER BY id DESC LIMIT 1"
    )
    p_cad = cur.fetchone()[0]

    # designações da Sentry (resumo mais recente)
    cur.execute(
        "SELECT payload FROM bronze.sentry_raw WHERE source = 'sentry_resumo' ORDER BY id DESC LIMIT 1"   # noqa: E501
    )
    p_sentry = cur.fetchone()[0]

i_des = p_cad["fields"].index("des")
des_cad = {linha[i_des] for linha in p_cad["data"]}
des_sentry = {r["des"] for r in p_sentry["data"]}

print("NeoWs:", len(des_neows), "| CAD:", len(des_cad), "| Sentry:", len(des_sentry))
print("NeoWs ∩ CAD:", len(des_neows & des_cad))
print("NeoWs ∩ Sentry:", len(des_neows & des_sentry))
print("CAD ∩ Sentry:", len(des_cad & des_sentry))
print("exemplos NeoWs:", list(des_neows)[:5])
print("exemplos CAD:", list(des_cad)[:5])

NeoWs: 34 | CAD: 33 | Sentry: 2251
NeoWs ∩ CAD: 1
NeoWs ∩ Sentry: 2
CAD ∩ Sentry: 3
exemplos NeoWs: ['2019 VF1', '2019 UN14', '2019 WT1', '2019 XS', '2004 XK14']
exemplos CAD: ['2019 XF2', '2024 RR14', '2025 YN', '2022 VB2', '2026 QQ8']


In [ ]:
od = lookup["orbital_data"]
print("orbital_data:", {k: type(v).__name__ for k, v in od.items()}, "\n")
print("orbit_class:", od["orbit_class"], "\n")
print("estimated_diameter:", lookup["estimated_diameter"], "\n")

# o feed é um subconjunto do histórico do lookup? (decide a fonte de aproximações)
with conn.cursor() as cur:
    cur.execute("SELECT payload FROM bronze.neows_lookup_raw")
    lookups = {
        p[0]["id"]: {c["close_approach_date"] for c in p[0]["close_approach_data"]}
        for p in cur.fetchall()
    }

total = faltando = 0
for _dia, lista in feed["near_earth_objects"].items():
    for ast in lista:
        if ast["id"] in lookups:
            for c in ast["close_approach_data"]:
                total += 1
                faltando += c["close_approach_date"] not in lookups[ast["id"]]

print("aproximações do feed cujo asteroide tem lookup:", total)
print("dessas, ausentes no histórico do lookup:", faltando)

orbital_data: {'equinox': 'str', 'orbit_id': 'str', 'inclination': 'str', 'mean_motion': 'str', 'orbit_class': 'dict', 'eccentricity': 'str', 'mean_anomaly': 'str', 'orbital_period': 'str', 'perihelion_time': 'str', 'semi_major_axis': 'str', 'data_arc_in_days': 'int', 'epoch_osculation': 'str', 'aphelion_distance': 'str', 'observations_used': 'int', 'orbit_uncertainty': 'str', 'perihelion_argument': 'str', 'perihelion_distance': 'str', 'last_observation_date': 'str', 'first_observation_date': 'str', 'ascending_node_longitude': 'str', 'orbit_determination_date': 'str', 'minimum_orbit_intersection': 'str', 'jupiter_tisserand_invariant': 'str'} 

orbit_class: {'orbit_class_type': 'APO', 'orbit_class_range': 'a (semi-major axis) > 1.0 AU; q (perihelion) < 1.017 AU', 'orbit_class_description': 'Near-Earth asteroid orbits which cross the Earth’s orbit similar to that of 1862 Apollo'} 

estimated_diameter: {'feet': {'estimated_diameter_max': 85.9063031073, 'estimated_diameter_min': 38.4184666